In [ ]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader, TensorDataset
import lightning as L

import matplotlib.pyplot as plt

# Tensor Basics -----------------------------------------------------------

# Create tensors
x = torch.arange(12, dtype=torch.float32).reshape((3,4))
y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

# Simple operations
addition = x + y
subtraction = x - y
multiplication = x * y
division = x / y

# Slicing
slicing = x[:, 1:3]

# Reshaping 
reshaping = x.reshape(4, 3)

# Broadcasting
broadcasting = x + torch.tensor([1, 2, 3, 4])

# Reduction
sum_reduction = x.sum()

# Applied functions
exponentiation = torch.exp(x)

# Dot product
u = torch.tensor([3.0, -4.0])
v = torch.tensor([2.0, 1.0])
dot_product = torch.dot(u, v)

# Matrix multiplication
matrix_multiplication = torch.matmul(x, y.t())  # .t() takes the transpose

# Norm
u_norm = torch.norm(u)

result_dict = {
    "Creation_x": x,
    "Creation_y": y,
    "Addition": addition,
    "Subtraction": subtraction,
    "Multiplication": multiplication,
    "Division": division,
    "Slicing": slicing,
    "Reshaping": reshaping,
    "Broadcasting": broadcasting,
    "Sum Reduction": sum_reduction,
    "Exponentiation": exponentiation,
    "Dot Product": dot_product,
    "Matrix Multiplication": matrix_multiplication,
    "Norm": u_norm
}

# Simulation of Neural Network -----------------------------------------------------------

# Define input data, weights, and 'true' output for loss calculation
input_tensor = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
weights = torch.tensor([[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]], requires_grad=True)
true_output = torch.tensor([[0.7, 1.5]])

# Forward pass - Matrix multiplication (input with weights) and a ReLU activation
layer_output = torch.mm(input_tensor, weights.t()).relu()

# Calculate loss - Mean Squared Error (MSE) 
predicted_output = layer_output
loss = (true_output - predicted_output).pow(2).mean()

# Backward pass - compute gradients with autograd
loss.backward()

# Display the tensors and gradients
print("Input Tensor:", input_tensor)
print("Weights:", weights)
print("Layer Output after ReLU:", layer_output)
print("Loss (MSE):", loss)
print("Gradient with respect to Input Tensor:", input_tensor.grad)
print("Gradient with respect to Weights:", weights.grad)

# Basic Neural Network - Just Pytorch ------------------------------------------------------

# Define neural network
class SimpleNN(torch.nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.weights = torch.nn.Parameter(torch.tensor([[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]))

    def forward(self, x):
        # Forward pass through layer and ReLU activation
        return torch.mm(x, self.weights.t()).relu()

# Instantiate neural network
model = SimpleNN()

# Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Define tensors for input and output 
input_tensor = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
true_output = torch.tensor([[0.7, 1.5]])

# Wrap tensors in a DataLoader
train_loader = DataLoader(TensorDataset(input_tensor, true_output), batch_size=1)

# Training loop
for epoch in range(1):  
    for batch in train_loader:
        # Separate data into inputs and outputs
        inputs, targets = batch
        # Zero gradients
        optimizer.zero_grad()
        # Forward pass
        predicted_output = model(inputs)
        # Compute loss
        loss = torch.mean((targets - predicted_output).pow(2))
        # Backward pass
        loss.backward()
        # Update weights
        optimizer.step()

# Basic Neural Network - Pytorch + Lightening ----------------------------------------------

class SimpleNN(L.LightningModule):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.weights = torch.nn.Parameter(torch.tensor([[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]))

    def forward(self, x):
        # Forward pass 
        return torch.mm(x, self.weights.t()).relu()

    def training_step(self, batch, batch_idx):
        # Forward pass with loss calculation
        x, true_output = batch
        predicted_output = self(x)
        loss = torch.mean((true_output - predicted_output).pow(2))
        return loss

    def configure_optimizers(self):
        # Optimization with SGD
        return torch.optim.SGD(self.parameters(), lr=0.01)

# Instantiate neural network
model = SimpleNN()

# Define input and output tensors 
input_tensor = torch.tensor([[1.0, 2.0, 3.0]])
true_output = torch.tensor([[0.7, 1.5]])

# Wrap data in a DataLoader
train_loader = DataLoader(TensorDataset(input_tensor, true_output), batch_size=1)

# Define trainer
trainer = L.Trainer(max_epochs=1)

# Fit model
trainer.fit(model, train_loader)

